<a href="https://colab.research.google.com/github/Hackathon-05-06-2026/Hackathon_Files/blob/main/Hcktn_R_(rf_lg).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# GOOGLE DRIVE CONNECTION
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# Example path:
# /content/drive/MyDrive/datasets/data.csv

DATASET_PATH = "/content/drive/MyDrive/datasets/data.csv"
TARGET_COLUMN = "target"  # Change this

# ==========================================
# IMPORTS
# ==========================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ==========================================
# RANDOM FOREST MODULE
# ==========================================
class RandomForestModule:

    def __init__(
        self,
        dataset_path,
        target_column,
        test_size=0.2,
        random_state=42
    ):
        self.dataset_path = dataset_path
        self.target_column = target_column
        self.test_size = test_size
        self.random_state = random_state

    def load_data(self):
        self.df = pd.read_csv(self.dataset_path)
        print(f"Dataset Shape: {self.df.shape}")
        return self.df

    def preprocess(self):

        df = self.df.copy()

        X = df.drop(columns=[self.target_column])
        y = df[self.target_column]

        # Encode categorical columns
        for col in X.select_dtypes(include=["object"]).columns:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))

        # Encode target if classification
        self.is_classification = (
            y.dtype == "object" or
            len(y.unique()) < 20
        )

        if self.is_classification:
            self.target_encoder = LabelEncoder()
            y = self.target_encoder.fit_transform(y)

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X,
            y,
            test_size=self.test_size,
            random_state=self.random_state
        )

        print("Preprocessing Complete")

    def train(self):

        if self.is_classification:

            self.model = RandomForestClassifier(
                n_estimators=200,
                random_state=self.random_state,
                n_jobs=-1
            )

        else:

            self.model = RandomForestRegressor(
                n_estimators=200,
                random_state=self.random_state,
                n_jobs=-1
            )

        self.model.fit(self.X_train, self.y_train)

        print("Training Complete")

    def evaluate(self):

        preds = self.model.predict(self.X_test)

        print("\n===== RESULTS =====")

        if self.is_classification:

            acc = accuracy_score(self.y_test, preds)

            print(f"Accuracy: {acc:.4f}")

            print("\nClassification Report:")
            print(classification_report(self.y_test, preds))

            print("\nConfusion Matrix:")
            print(confusion_matrix(self.y_test, preds))

        else:

            mse = mean_squared_error(self.y_test, preds)
            rmse = np.sqrt(mse)
            mae = mean_absolute_error(self.y_test, preds)
            r2 = r2_score(self.y_test, preds)

            print(f"MAE : {mae:.4f}")
            print(f"RMSE: {rmse:.4f}")
            print(f"R²  : {r2:.4f}")

    def feature_importance(self):

        importance = pd.DataFrame({
            "Feature": self.X_train.columns,
            "Importance": self.model.feature_importances_
        })

        importance = importance.sort_values(
            by="Importance",
            ascending=False
        )

        print("\nTop Features:")
        print(importance.head(20))

        return importance

    def run(self):

        self.load_data()
        self.preprocess()
        self.train()
        self.evaluate()
        return self.feature_importance()


# ==========================================
# EXECUTION
# ==========================================

rf = RandomForestModule(
    dataset_path=DATASET_PATH,
    target_column=TARGET_COLUMN
)

feature_importance = rf.run()

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/datasets/data.csv'